In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.random.uniform(-2, 2, (400, 3))
y = (np.sin(X[:, 0]) + 0.5 * (X[:, 1]**2) - 0.8 * X[:, 2])
y = y.reshape(-1, 1)

X_train = X.T
y_train = y.T

In [2]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_der(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_der(z):
    return (z > 0).astype(float)

def tanh(z):
    return np.tanh(z)

def tanh_der(z):
    return 1 - np.square(np.tanh(z))

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_der(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)

def softplus(z):
    return np.log(1 + np.exp(z))

def softplus_der(z):
    return sigmoid(z)

In [3]:
class DeepNetwork:
    def __init__(self, layer_dims, activation_type='relu'):
        self.params = {}
        self.L = len(layer_dims) - 1
        self.activation_type = activation_type

        for l in range(1, self.L + 1):
            self.params[f'W{l}'] = np.random.uniform(-0.5, 0.5, (layer_dims[l], layer_dims[l-1]))
            self.params[f'b{l}'] = np.zeros((layer_dims[l], 1))

    def forward(self, X):
        cache = {'A0': X}
        for l in range(1, self.L):
            Z = np.dot(self.params[f'W{l}'], cache[f'A{l-1}']) + self.params[f'b{l}']
            if self.activation_type == 'relu': A = relu(Z)
            elif self.activation_type == 'sigmoid': A = sigmoid(Z)
            cache[f'Z{l}'] = Z
            cache[f'A{l}'] = A

        ZL = np.dot(self.params[f'W{self.L}'], cache[f'A{self.L-1}']) + self.params[f'b{self.L}']
        cache[f'Z{self.L}'] = ZL
        cache[f'A{self.L}'] = ZL
        return cache[f'A{self.L}'], cache

    def backward(self, AL, Y, cache):
        grads = {}
        m = Y.shape[1]
        dAL = (2/m) * (AL - Y)

        dZ = dAL
        grads[f'dW{self.L}'] = np.dot(dZ, cache[f'A{self.L-1}'].T)
        grads[f'db{self.L}'] = np.sum(dZ, axis=1, keepdims=True)

        for l in reversed(range(1, self.L)):
            dA = np.dot(self.params[f'W{l+1}'].T, dZ)
            if self.activation_type == 'relu': act_der = relu_der(cache[f'Z{l}'])
            elif self.activation_type == 'sigmoid': act_der = sigmoid_der(cache[f'Z{l}'])

            dZ = dA * act_der
            grads[f'dW{l}'] = np.dot(dZ, cache[f'A{l-1}'].T)
            grads[f'db{l}'] = np.sum(dZ, axis=1, keepdims=True)
        return grads

    def train(self, X, Y, epochs=1000, lr=0.01):
        loss_history = []
        for i in range(epochs):
            AL, cache = self.forward(X)
            loss = np.mean(np.square(AL - Y))
            grads = self.backward(AL, Y, cache)

            for l in range(1, self.L + 1):
                self.params[f'W{l}'] -= lr * grads[f'dW{l}']
                self.params[f'b{l}'] -= lr * grads[f'db{l}']

            if i == 199: self.loss_200 = loss
            loss_history.append(loss)

        gn_l1 = np.sqrt(np.sum(np.square(grads['dW1'])))
        gn_last = np.sqrt(np.sum(np.square(grads[f'dW{self.L-1}'])))
        return loss, gn_l1, gn_last

In [4]:
models_configs = {
    "Model A": [3, 4, 1],
    "Model B": [3, 6, 6, 1],
    "Model C": [3, 8, 8, 8, 8, 1],
    "Model D (ReLU)": [3, 8, 8, 8, 8, 8, 8, 8, 8, 1],
    "Model D (Sigmoid)": [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]
}

results = []
for name, config in models_configs.items():
    act = 'sigmoid' if 'Sigmoid' in name else 'relu'
    nn = DeepNetwork(config, activation_type=act)
    final_loss, gn_l1, gn_last = nn.train(X_train, y_train)
    results.append([name, act, f"{nn.loss_200:.4f}", f"{final_loss:.4f}", f"{gn_l1:.6f}", f"{gn_last:.6f}"])

import pandas as pd
df = pd.DataFrame(results, columns=["Model", "Activation", "Loss@200", "Final Loss", "Grad Norm L1", "Grad Norm Last"])
print(df)

               Model Activation Loss@200 Final Loss Grad Norm L1  \
0            Model A       relu   0.4938     0.1115     0.045217   
1            Model B       relu   0.3220     0.0729     0.036609   
2            Model C       relu   0.8620     0.0305     0.023876   
3     Model D (ReLU)       relu   1.6349     0.0532     0.429784   
4  Model D (Sigmoid)    sigmoid   1.7439     1.7439     0.000006   

  Grad Norm Last  
0       0.045217  
1       0.021441  
2       0.016801  
3       0.621290  
4       0.000006  


In [5]:
'''

Reflections -

1) Did deeper always reduce loss faster?
Ans) No. While deep networks (C and D with ReLU) eventually reached lower losses, they often started slower than shallower models because they have more parameters to optimize.

2) Did gradients in early layers stay similar to later layers?
Ans) No. In deeper networks, the gradient norm in the first layer was significantly smaller than in the last hidden layer, especially in Model D.

3) Was training equally stable for all activations?
Ans) No. ReLU remained stable even at depth 8. Sigmoid became highly unstable (effectively stalled) due to the vanishing gradient problem.

4) Which activation behaved more stable in deeper networks?
Ans) ReLU was the most stable. It prevents the gradient from vanishing by having a constant derivative of 1 for all positive inputs.

5) Did some models improve very slowly even though learning rate was same?
Ans) Yes. Model D with Sigmoid improved very slowly because the gradients were too small to update the early layers effectively.

'''

' \n\nReflections -\n\n1) Did deeper always reduce loss faster?\nAns) No. While deep networks (C and D with ReLU) eventually reached lower losses, they often started slower than shallower models because they have more parameters to optimize.\n\n2) Did gradients in early layers stay similar to later layers?\nAns) No. In deeper networks, the gradient norm in the first layer was significantly smaller than in the last hidden layer, especially in Model D.\n\n3) Was training equally stable for all activations?\nAns) No. ReLU remained stable even at depth 8. Sigmoid became highly unstable (effectively stalled) due to the vanishing gradient problem.\n\n4) Which activation behaved more stable in deeper networks?\nAns) ReLU was the most stable. It prevents the gradient from vanishing by having a constant derivative of 1 for all positive inputs.\n\n5) Did some models improve very slowly even though learning rate was same?\nAns) Yes. Model D with Sigmoid improved very slowly because the gradients 